# Visualización de polígonos del Observatorio

**Tarea 1.3 - Fase 1**

Este notebook carga `config/poligonos.geojson` y genera una visualización interactiva sobre un mapa base de Posadas.
Sirve para validar manualmente que los polígonos cubren las zonas de interés y no se solapan indebidamente.

Fecha: 2026-04-22

In [ ]:
# Imports - geopandas para IO, folium para mapa interactivo
from pathlib import Path

import geopandas as gpd
import folium

# ipyleaflet es opcional; si no está, seguimos con folium.
try:
    import ipyleaflet  # noqa: F401
    _TIENE_IPYLEAFLET = True
except ImportError:
    _TIENE_IPYLEAFLET = False
    print('ipyleaflet no disponible, se usa folium (suficiente para validación).')

In [ ]:
# Cargar polígonos de interés
RUTA_GEOJSON = Path('../config/poligonos.geojson')
gdf = gpd.read_file(RUTA_GEOJSON)
print(f'CRS origen: {gdf.crs}')
print(f'Cantidad de polígonos: {len(gdf)}')
gdf.head()

In [ ]:
# Estadísticas básicas: cantidad por categoría y área en km² (reproyectada a EPSG:32721 - UTM 21S)
gdf_metrico = gdf.to_crs('EPSG:32721')
gdf_metrico['area_km2'] = gdf_metrico.geometry.area / 1_000_000.0

print('--- Por categoría ---')
print(gdf['categoria'].value_counts())
print()
print('--- Áreas por polígono (km²) ---')
resumen = gdf_metrico[['id', 'nombre', 'categoria', 'prioridad', 'area_km2']].copy()
resumen['area_km2'] = resumen['area_km2'].round(3)
resumen.sort_values('area_km2', ascending=False)

In [ ]:
# Mapa folium centrado en Posadas con layer de polígonos
# Colores por categoría
COLOR_POR_CATEGORIA = {
    'asentamiento_crecimiento_rapido': '#c97d3c',
    'consolidado_crecimiento': '#1a3a5c',
    'control_consolidado': '#5a7a9c',
    'zona_sensible': '#8b3a3a',
}

CENTRO_POSADAS = (-27.3667, -55.8967)
mapa = folium.Map(location=CENTRO_POSADAS, zoom_start=12, tiles='OpenStreetMap')

def estilo_feature(feature):
    cat = feature['properties'].get('categoria', '')
    color = COLOR_POR_CATEGORIA.get(cat, '#666666')
    return {
        'fillColor': color,
        'color': color,
        'weight': 2,
        'fillOpacity': 0.35,
    }

folium.GeoJson(
    gdf.__geo_interface__,
    name='Polígonos Observatorio',
    style_function=estilo_feature,
    tooltip=folium.GeoJsonTooltip(
        fields=['nombre', 'categoria', 'prioridad'],
        aliases=['Nombre:', 'Categoría:', 'Prioridad:'],
    ),
    popup=folium.GeoJsonPopup(
        fields=['id', 'nombre', 'descripcion', 'categoria'],
        aliases=['ID:', 'Nombre:', 'Descripción:', 'Categoría:'],
        localize=True,
    ),
).add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa

In [ ]:
# Export opcional a HTML para adjuntar al reporte
SALIDA_HTML = Path('../data/outputs/poligonos_mapa.html')
SALIDA_HTML.parent.mkdir(parents=True, exist_ok=True)
mapa.save(str(SALIDA_HTML))
print(f'Mapa guardado en: {SALIDA_HTML.resolve()}')

## Checklist de validación visual

Revisar antes de dar por válido el archivo `config/poligonos.geojson`:

- [ ] ¿Cada polígono cubre el área correcta del barrio objetivo (comparar con Google Maps)?
- [ ] ¿Ningún polígono solapa con otro de forma significativa (overlap < 5%)?
- [ ] ¿Los polígonos están dentro del bbox de Posadas declarado en `settings.yaml`?
- [ ] ¿La categoría declarada coincide con lo que se ve (control consolidado vs. crecimiento)?
- [ ] ¿El polígono sensible (El Brete) tiene `sensible=true` y `publicar_en_sitio=false`?
- [ ] ¿Las áreas en km² son razonables (entre 0.5 y 10 km² para barrios de Posadas)?

Si algún ítem falla, editar `config/poligonos.geojson` y re-ejecutar este notebook.